# Notebook 04 — RAG Pipeline

**User-owned notebook — primary RAG exploration phase.**

**Prerequisites:**
1. Ollama running: `ollama serve`
2. Models pulled: `ollama pull llama3 && ollama pull nomic-embed-text`
3. Notebook 03 completed (parsers + chunker verified)

**What to explore here:**
- Ollama health check
- Embed a string
- Ingest 2-3 real policy docs end-to-end
- Inspect ChromaDB collection
- Raw similarity search
- Role-filtered search (same query + different role sets = different results)
- Archived toggle test

In [ ]:
import sys
sys.path.insert(0, '..')

import uuid
from pathlib import Path

from policy_system.llm.ollama_provider import OllamaProvider
from policy_system.rag.chromadb_provider import ChromaDBProvider
from policy_system.ingestion.pipeline import ingest_document, parse_document, chunk_document
from policy_system.ingestion.parsers.base import ParsedDocument, ParsedPage

# Providers
llm = OllamaProvider()
rag = ChromaDBProvider(persist_dir='../data/chroma_db', collection_name='notebook_04_test')

print('Providers initialized')

## 1. Ollama Health Check

In [ ]:
healthy = llm.health_check()
print(f'Ollama health: {"OK" if healthy else "FAILED"}')
if not healthy:
    print('Make sure ollama is running and models are pulled:')
    print('  ollama serve')
    print('  ollama pull llama3')
    print('  ollama pull nomic-embed-text')

## 2. Embed a String

In [ ]:
test_text = 'All employees must use MFA when accessing company systems.'
embedding = llm.embed(test_text)
print(f'Embedding dimensions: {len(embedding)}')
print(f'First 5 values: {embedding[:5]}')
assert len(embedding) > 100, 'Embedding should have >100 dimensions'

## 3. Create Synthetic Policy Documents for Testing

In [ ]:
# Create two synthetic policy docs — one IT, one Finance
uploads_dir = Path('../data/uploads')
uploads_dir.mkdir(parents=True, exist_ok=True)

it_policy_text = """IT Security Policy

Section 1: Access Control

All employees must use multi-factor authentication (MFA) when accessing company systems. Passwords must be at least 12 characters and changed every 90 days. Shared credentials are strictly prohibited.

Section 2: Data Classification

Company data is classified as Public, Internal, Confidential, or Restricted. Restricted data must be encrypted at rest and in transit. Access to Restricted data requires manager approval and is logged.

Section 3: Incident Response

Security incidents must be reported to the IT Security team within 1 hour of discovery. The incident response team will assess severity and initiate containment procedures. All incidents are logged and reviewed quarterly.
"""

finance_policy_text = """Finance Compliance Policy

Section 1: Expense Reporting

All employee expenses must be submitted within 30 days of incurrence. Expenses over $500 require manager approval. Receipts are mandatory for all expenses over $25. Corporate cards must not be used for personal expenses.

Section 2: Budget Management

Department budgets are approved annually by the CFO. Budget transfers between cost centers require Finance team approval. Unplanned expenditures over 10% of quarterly budget must be escalated to the Finance Director.

Section 3: Vendor Payments

All vendor invoices must be approved by two signatories over $10,000. Payment terms are net-30 by default. Early payment discounts require CFO authorization. All vendor contracts must be reviewed by Legal before signing.
"""

(uploads_dir / 'it_security_policy.txt').write_text(it_policy_text)
(uploads_dir / 'finance_compliance.txt').write_text(finance_policy_text)
print('Created synthetic policy documents')

## 4. Ingest Documents End-to-End

In [ ]:
# Role IDs — in real usage these come from the SQL roles table
IT_ROLE_ID = 'role-it-eng-00000000000000000000'
FINANCE_ROLE_ID = 'role-finance-000000000000000000000'
AUDITOR_ROLE_ID = 'role-global-audit-00000000000000000'

IT_DOC_ID = str(uuid.uuid4())
FINANCE_DOC_ID = str(uuid.uuid4())

# Ingest IT doc (accessible to IT engineers and auditors)
it_result = ingest_document(
    file_path='../data/uploads/it_security_policy.txt',
    doc_id=IT_DOC_ID,
    allowed_roles=[IT_ROLE_ID, AUDITOR_ROLE_ID],
    rag_provider=rag,
    llm_provider=llm,
    title='IT Security Policy',
)
print(f'IT doc ingested: {it_result}')

# Ingest Finance doc (accessible to finance and auditors)
fin_result = ingest_document(
    file_path='../data/uploads/finance_compliance.txt',
    doc_id=FINANCE_DOC_ID,
    allowed_roles=[FINANCE_ROLE_ID, AUDITOR_ROLE_ID],
    rag_provider=rag,
    llm_provider=llm,
    title='Finance Compliance Policy',
)
print(f'Finance doc ingested: {fin_result}')

print(f'\nTotal chunks in ChromaDB: {rag.get_chunk_count()}')

## 5. Raw Similarity Search (no role filter)

In [ ]:
query = 'What are the password requirements?'
query_embedding = llm.embed(query)

# Unfiltered search — use auditor role which has access to all
results = rag.similarity_search(
    query_embedding=query_embedding,
    allowed_role_ids=[AUDITOR_ROLE_ID],
    top_k=3,
    include_archived=False,
)

print(f'Query: "{query}"')
print(f'Results: {len(results)}')
for i, r in enumerate(results, 1):
    print(f'\n[{i}] Score={r.score:.3f} | {r.doc_title} | Page {r.page_number}, Para {r.para_number}')
    print(f'    {r.text[:200]}...')

## 6. Role-Filtered Search — Security Test

In [ ]:
query = 'What are the approval requirements?'
query_embedding = llm.embed(query)

# IT user search — should only see IT docs
it_results = rag.similarity_search(
    query_embedding=query_embedding,
    allowed_role_ids=[IT_ROLE_ID],
    top_k=5,
)
print('IT user results:')
for r in it_results:
    print(f'  [{r.doc_title}] Score={r.score:.3f}')
    assert r.doc_id == IT_DOC_ID, f'IT user should not see finance docs! Got: {r.doc_title}'

print()

# Finance user search — should only see Finance docs
fin_results = rag.similarity_search(
    query_embedding=query_embedding,
    allowed_role_ids=[FINANCE_ROLE_ID],
    top_k=5,
)
print('Finance user results:')
for r in fin_results:
    print(f'  [{r.doc_title}] Score={r.score:.3f}')
    assert r.doc_id == FINANCE_DOC_ID, f'Finance user should not see IT docs! Got: {r.doc_title}'

print('\nSECURITY TEST PASSED: Role isolation is working correctly')

## 7. Archive Toggle Test

In [ ]:
query_embedding = llm.embed('password requirements')

# Archive the IT doc
rag.update_archived_status(IT_DOC_ID, is_archived=True)
print('IT doc archived')

# IT user searches WITHOUT include_archived — should get 0 results
results_no_archive = rag.similarity_search(
    query_embedding=query_embedding,
    allowed_role_ids=[IT_ROLE_ID],
    top_k=5,
    include_archived=False,
)
print(f'Without include_archived: {len(results_no_archive)} results')
assert len(results_no_archive) == 0

# IT user searches WITH include_archived — should get results
results_with_archive = rag.similarity_search(
    query_embedding=query_embedding,
    allowed_role_ids=[IT_ROLE_ID],
    top_k=5,
    include_archived=True,
)
print(f'With include_archived: {len(results_with_archive)} results')
assert len(results_with_archive) > 0

# Restore
rag.update_archived_status(IT_DOC_ID, is_archived=False)
print('IT doc unarchived')
print('Archive toggle test: PASS')

## Summary

RAG pipeline is working end-to-end:
- Ollama health check ✓
- Embedding via nomic-embed-text ✓
- Document ingestion (parse → chunk → embed → store) ✓
- ChromaDB collection inspection ✓
- Role-filtered similarity search with **pre-filter security** ✓
- Archive toggle (include/exclude) ✓

**Next:** Notebook 05 — Query Engine (end-to-end query with citations)